In [ ]:
# ==============================================================================
# 🚀 KAGGLE NOTEBOOK: TRANSCRIÇÃO AUTOMÁTICA DOS 82 ÁUDIOS FALTANTES
# ==============================================================================
import os
import sys
import re
import json
import subprocess
from pathlib import Path

# 1. Instalar dependências no ambiente Linux do Kaggle
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)
subprocess.run("curl -s https://rclone.org/install.sh | bash", shell=True, check=True)

from faster_whisper import WhisperModel

# 2. Configurar Rclone para o Google Drive com o Token OAuth Atualizado
RCLONE_CONFIG_DIR = Path("/root/.config/rclone")
RCLONE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

RCLONE_CONF_CONTENT = """[meudrive]
type = drive
scope = drive
token = {"access_token":"YOUR_ACCESS_TOKEN","token_type":"Bearer","refresh_token":"YOUR_REFRESH_TOKEN"}
"""

with open(RCLONE_CONFIG_DIR / "rclone.conf", "w", encoding="utf-8") as f:
    f.write(RCLONE_CONF_CONTENT)

# 3. Configurar caminhos temporários no Kaggle
KAGGLE_AUDIOS_DIR = Path("/kaggle/working/audios")
KAGGLE_TXT_DIR = Path("/kaggle/working/transcricoes_txt")
KAGGLE_JSON_DIR = Path("/kaggle/working/transcricoes_json")

KAGGLE_AUDIOS_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_TXT_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_JSON_DIR.mkdir(parents=True, exist_ok=True)

REMOTE_PODCASTS = "meudrive:IBPM_CR_Cortes/audio_podcasts"
REMOTE_P06 = "meudrive:IBPM_CR_Cortes/06_Podcasts_Audio"
REMOTE_SAIDA_TXT = "meudrive:IBPM_CR_Cortes/transcricoes_whisper_txt"
REMOTE_SAIDA_JSON = "meudrive:IBPM_CR_Cortes/transcricoes_whisper_json"

# 4. LISTA ALVO EXATA DOS ÁUDIOS FALTANTES
LISTA_EXATA_FALTANTES = [
  "001_2hvx5L2DR2U_culto_santa_ceia_dia_02_10_2022.webm",
  "003_c9apu4Aormc_culto_de_celebracao_09_10.webm",
  "021_OleOihvb8Wk_quinta_profetica_15_12_2022.webm",
  "022_1-J9sdxbs04_culto_de_natal_18_12_2022.webm",
  "025_83jIjuJMEk0_quinta_profetica_05_01_2022.webm",
  "028_MaL7VfjzxnY_domingo_de_celebracao_15_01_2023.webm",
  "045_rhVKtUpgx7w_quinta_profetica_30_03_2023.webm",
  "048_QfVsYv4vzEA_quinta_profetica_da_redencao_06_04_2023.webm",
  "049_g4W4CMMonu0_domingo_culto_de_pascoa_09_04_2023.webm",
  "050_NYr_VS-wKVA_Domingo_Culto_de_P_scoa__09_04_2023.webm",
  "052_UxchF5M1TYg_domingo_de_celebracao_16_04_2023.webm",
  "057_p5i1LCvzWvE_quinta_profetica_da_familia_04_05_2023.webm",
  "059_2hhL-Tq2QiY_domingo_santa_ceia_07_05_2023.webm",
  "060_yNWViCaW1_4_Domingo_Santa_Ceia__07_05_2023.webm",
  "062_2QtNB1kvXng_domingo_de_celebracao_dia_das_maes_14_05_2023.webm",
  "064_hNeM9ic71R0_quinta_profetica_da_familia_18_05_2023.webm",
  "065_7MuaOrZS53Y_domingo_celebracao_da_familia_21_05_2023.webm",
  "073_ikGrfwaIiNU_quinto_dia_de_festividade_08_07.webm",
  "082_uhJEt7ZBwXg_domingo_de_santa_ceia_06_08_23.webm",
  "084_8QDErFAO1zc_culto_dia_dos_pais_13_08_23.webm",
  "085_iBFl3QCw05A_culto_dia_dos_pais_13_08_23.webm",
  "086_GDb6dIji6dg_quinta_feira_profetica_17_08_23.webm",
  "088_NTEJejlgBi8_quinta_feira_profetica_24_08_23.webm",
  "089_Yb_6_tZUmWI_DOMINGO_DE_CELEBRA__O__27_08_23.webm",
  "090_UhtkIwXfbyk_quinta_profetica_31_08_23.webm",
  "101_9n98OHAaBMc_terceiro_dia_de_re_festa_07_10_23.webm",
  "102_mhtZQdR_mZs_QUARTO_DIA_DE_RE_FESTA__08_10_23.webm",
  "107_6gMIVFNV7m8_quinta_profetica_26_10_23.webm",
  "111_38xbrmz7e50_quinta_profetica_09_11_23.webm",
  "114_V4oH86ItMB8_domingo_de_celebracao_19_11_23.webm",
  "117_kALO6xBj1Wk_quinta_profetica_30_11_23.webm",
  "122_dYy1edFhw64_domingo_culto_de_natal_17_12_23.webm",
  "126_sTKABTeqC7s_domingo_culto_da_virada_2024_31_12_23.webm",
  "127_wMtdeSP_PBk_QUINTA_PROF_TICA___AVIVAMENTO_E_INTIMIDADE__04_01_23.webm",
  "133_LbAbxxpicec_domingo_de_celebracao_21_01_24.webm",
  "143_RZeH_28hnQw_MINI_VIG_LIA___AVIVAMENTO_E_INTIMIDADE__23_02_24.webm",
  "146_NddisUFEUyI_domingo_santa_ceia_03_03_24.webm",
  "154_eCrTjaH1j7I_domingo_de_celebracao_culto_de_pascoa_31_03_24.webm",
  "158_kSxBUPt9Bvg_quinta_profetica_derrubando_as_mulharas_da_minha_vida_11_04.webm",
  "162_lcQfI-svRrA_domingo_de_celebracao_21_04_24.webm",
  "167_TTsj3E_ebLI_CULTO_DAS_P_ROLAS____08_05_24.webm",
  "175_ILzAvDqYI8Y_quinta_profetica_rompendo_limites_30_05_24.webm",
  "217_s83xhQn4J_Q_DOMINGO_DE_SANTA_CEIA___INTIMIDADE__06_10_24.webm",
  "225_jfE4kRU7Mrc_quinta_profetica_teu_nome_e_cura_24_10_24.webm",
  "227_60Mn7cT3_nE_QUINTA_PROF_TICA___TEU_NOME___CURA__31_10_24.webm",
  "237_MK6w25_MKXw_QUINTA_PROF_TICA_DA_FAM_LIA____05_12_24.webm",
  "248_o-CAO5cQPDQ_quinta_profetica_02_01_24.m4a",
  "265_UR11tNNKReM_domingo_de_celebracao_02_03_25.m4a",
  "269_ebNluIDhd90_quarta_profetica_chame_a_existencia_12_03_25.m4a",
  "272_fSG96TmMUk4_quarta_profetica_chame_a_existencia_19_03_25.m4a",
  "277_vIEQEx6KisA_primeiro_dia_conferencia_01_04_25.m4a",
  "284_qAMlfgJlqKI_culto_de_pascoa_20_04_25.m4a",
  "288_R8GgaByob8g_culto_de_santa_ceia_04_05_25.m4a",
  "289_NNVPhI5GTRA_culto_de_santa_ceia_04_05_25.m4a",
  "299_EVldzg5G1Os_quarta_profetica_atos_2_04_06_25.m4a",
  "306_Wg5qbNI_X-w_QUARTA_PROF_TICA___FAZ_DE_NOVO__02_07_25.m4a",
  "309_YQPyEe19uAw_2_dia_festividade_maa_09_07_25.m4a",
  "321_RtFG7tRir9I_quarta_profetica_faz_de_novo_30_07_25.m4a",
  "325_WBn23cPHELo_quarta_profetica_alegrai_vos_13_08_25.m4a",
  "357_pIyDow9iaZs_quarta_profetica_o_desafio_da_fe_12_11_25.m4a",
  "373_1KvwI8L7Um4_quarta_profetica_efata_07_01_26.m4a",
  "380_yq3Vcl5zl1I_quarta_profetica_efata_28_01_26.m4a",
  "381_2siKjuEmpq0_domingo_de_celebracao_15_02_26.m4a",
  "386_gFeXeSlsEuE_quarta_profetica_esforca_te_04_03_26.m4a",
  "394_ECFjGc3049g_domingo_de_celebracao_29_03_26.m4a",
  "397_o2J7qjqheSo_quarta_profetica_a_cruz_08_04_26.m4a",
  "398_H8Q3dsXdLlI_domingo_de_celebracao_12_04_26.m4a",
  "405_19N573Txx0w_quarta_profetica_a_cruz_29_04_26.m4a",
  "408_ZbiTyRI2PHQ_domingo_santa_ceia_03_05_26.m4a",
  "421_CRDG6bnhBXA_domingo_de_celebracao_31_05_26.m4a",
  "428_eUlr3RNeNsE_domingo_de_celebracao_21_06_26.m4a",
  "433_Izk3my2j3uQ_domingo_de_celebracao_12_07_26.m4a",
  "434_IaqUSzEzuxo_quarta_profetica_restituicao_15_07_26.m4a",
  "435_XqLuz7HRv_M_MINI_VIG_LIA___REFORMANDO_O_ALTAR__17_07_26.m4a",
  "438_5t26RhzBOA0_domingo_sala_de_adoracao_26_07_26.m4a",
  "440_5NwIiPBdVfQ_domingo_sala_de_adoracao_26_07_26.m4a",
  "443_nDSulaP76b8_domingo_de_celebracao_09_08_26.m4a",
  "448__dFs9v1PIHc_3__DIA_DE_FESTIVIDADE___MULHERES__13_08_26.m4a",
  "452_Hq7bcchLSms_5_dia_de_festividade_homens_14_08_26.m4a",
  "453_yLOGTSV6i0g_5_dia_de_festividade_homens_15_08_26.m4a",
  "455_mJn9p2a9xWs_6_dia_de_festividade_encerramento_16_08_26.m4a"
]

print("==============================================================")
print("TRANSCRIÇÃO ACELERADA EM GPU T4 (KAGGLE) VIA FASTER-WHISPER")
print(f"TARGET: EXATAMENTE {len(LISTA_EXATA_FALTANTES)} ÁUDIOS FALTANTES")
print("==============================================================\n", flush=True)

# 5. Sincronizar os áudios específicos do Drive para o Kaggle
print("Sincronizando os audios alvos do Google Drive para o Kaggle...", flush=True)
subprocess.run(["rclone", "copy", REMOTE_PODCASTS, str(KAGGLE_AUDIOS_DIR)])
subprocess.run(["rclone", "copy", REMOTE_P06, str(KAGGLE_AUDIOS_DIR)])

def extract_video_id(filename: str):
    m = re.search(r'_([a-zA-Z0-9_-]{11})_', filename) or re.search(r'^([a-zA-Z0-9_-]{11})', filename)
    return m.group(1) if m else None

set_alvos = set(LISTA_EXATA_FALTANTES)
set_ids_alvos = set(filter(None, [extract_video_id(name) for name in LISTA_EXATA_FALTANTES]))

audios_locais = sorted(list(KAGGLE_AUDIOS_DIR.glob("*")))
audios_pendentes = []

for arq in audios_locais:
    if arq.is_dir():
        continue
    vid = extract_video_id(arq.name)
    if arq.name in set_alvos or (vid and vid in set_ids_alvos):
        caminho_txt = KAGGLE_TXT_DIR / f"{arq.stem}.txt"
        if not caminho_txt.exists():
            audios_pendentes.append(arq)

print(f"Audios pendentes identificados para processamento: {len(audios_pendentes)}", flush=True)

# 6. Carregar Modelo Faster-Whisper na GPU CUDA
MODEL_SIZE = "medium"
print(f"\nCarregando modelo Faster-Whisper '{MODEL_SIZE}' na GPU CUDA...", flush=True)
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float32")
print("Modelo carregado com sucesso na GPU T4!", flush=True)

# 7. Processar os áudios alvos na GPU T4
sucessos = 0
erros = 0

for idx, arq in enumerate(audios_pendentes, start=1):
    nome_stem = arq.stem
    caminho_txt = KAGGLE_TXT_DIR / f"{nome_stem}.txt"
    caminho_json = KAGGLE_JSON_DIR / f"{nome_stem}.json"
    
    print(f"\n[{idx}/{len(audios_pendentes)}] Transcrevendo na GPU T4: {arq.name}...", flush=True)
    
    try:
        segments, info = model.transcribe(str(arq), language="pt", beam_size=5)
        
        texto_completo = []
        segmentos_json = []
        
        for segment in segments:
            texto_completo.append(segment.text.strip())
            segmentos_json.append({
                "start": round(segment.start, 2),
                "end": round(segment.end, 2),
                "text": segment.text.strip()
            })
            
        texto_final = " ".join(texto_completo)
        words_count = len(texto_final.split())
        
        with open(caminho_txt, "w", encoding="utf-8") as f_txt:
            f_txt.write(texto_final)
            
        with open(caminho_json, "w", encoding="utf-8") as f_json:
            json.dump({
                "file": arq.name,
                "language": info.language,
                "duration": round(info.duration, 2),
                "words_count": words_count,
                "text": texto_final,
                "segments": segmentos_json
            }, f_json, ensure_ascii=False, indent=2)
            
        print(f"   Concluido em GPU T4! Duracao: {info.duration:.1f}s | Palavras: {words_count}")
        
        subprocess.run(["rclone", "copy", str(caminho_txt), REMOTE_SAIDA_TXT], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(["rclone", "copy", str(caminho_json), REMOTE_SAIDA_JSON], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"   Sincronizado no Google Drive!", flush=True)
        
        sucessos += 1
        
    except Exception as err:
        print(f"   Erro ao transcrever {arq.name}: {err}", flush=True)
        erros += 1

print(f"\n===============================================================")
print(f"TRANSCRIÇÃO NO KAGGLE FINALIZADA!")
print(f"   • Sucessos: {sucessos}")
print(f"   • Erros:    {erros}")
print(f"===============================================================", flush=True)
